# DEM and Gaussian Process Workflow

## 1. Setup and Run Settings

In [ ]:
import importlib.util
import subprocess
import sys
packages = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "numba": "numba", "joblib": "joblib", "threadpoolctl": "threadpoolctl"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import sys
import os
import json
import time
import math
import hashlib
import warnings
import zipfile
import io
import base64
import shutil
import importlib.util
from pathlib import Path
from dataclasses import dataclass, asdict

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from scipy.stats import qmc, norm
from scipy.special import log_ndtr
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score
from threadpoolctl import threadpool_limits
from IPython.display import display
import joblib

try:
    from numba import njit
except ImportError:
    def njit(*args, **kwargs):
        return lambda f: f

RUN_MODE = "small_test"
#use new_search for the the best KE ratio(whole experiment)
RUN_SENSITIVITY = False
RECHECK_SELECTED_DESIGNS = True
RUN_PREDICTION_TEST = False
RUN_NEXT_STEP = False
USE_GOOGLE_DRIVE = False
DOWNLOAD_RESULTS = True
DEM_ZIP_PATH = ""

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Week6_DEM_GP") / RUN_MODE
else:
    OUTPUT_ROOT = Path("/content/Week6_DEM_GP" if Path("/content").exists() else "Week6_DEM_GP") / RUN_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES = OUTPUT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

CATEGORIES = ("homogeneous", "heterogeneous", "nonbistable")
BASE_C2 = np.array([-70., -100., -5., -35.])
BASE_C3 = np.array([1200., 2000., 1500., 3000.])
PERTURBATIONS = np.linspace(-0.5, 0.5, 21)
PAPER_RATIOS = {"homogeneous": 0.0117, "heterogeneous": 0.0056, "nonbistable": 0.0086}
SLIDE_GP_RATIOS = {"homogeneous": 0.009607644401963659,
                   "heterogeneous": 0.004482937563753115,
                   "nonbistable": 0.007126559400167516}
CATEGORY_TITLES = {"homogeneous": "Homogeneous", "heterogeneous": "Heterogeneous",
                   "nonbistable": "Heterogeneous excluding bistability"}
INITIAL_DESIGNS = 256
FIRST_GP_ROUNDS = 40
FIRST_GP_BATCH = 16
FIRST_GP_SEEDS = (2026, 2027, 2028)
EXTENDED_ROUNDS = 180
N_REGIONS = 6
REGION_NEIGHBORS = 40
REGION_ROUNDS = 35
REGION_BATCH = 7
REGION_SEPARATION = 0.3
REGION_INITIAL_SD = 0.04
REGION_INITIAL_RADIUS = 0.07
CONTINUE_REGIONS = 3
CONTINUE_ROUNDS = 35
NEXT_INITIAL = 64
NEXT_ROUNDS = 60
NEXT_BATCH = 6

if RUN_MODE not in ("saved_presentation", "new_search", "small_test"):
    raise ValueError("RUN_MODE must be saved_presentation, new_search, or small_test.")
if RUN_MODE == "small_test":
    CATEGORIES = ("heterogeneous",)
    INITIAL_DESIGNS = 16
    FIRST_GP_ROUNDS = 1
    FIRST_GP_BATCH = 4
    FIRST_GP_SEEDS = (2026,)
    EXTENDED_ROUNDS = 1
    N_REGIONS = 2
    REGION_NEIGHBORS = 2
    REGION_ROUNDS = 1
    CONTINUE_REGIONS = 1
    CONTINUE_ROUNDS = 1
    PERTURBATIONS = np.array([-0.05, 0.0, 0.05])
    NEXT_INITIAL = 8
    NEXT_ROUNDS = 1

@dataclass(frozen=True)
class SolverSettings:
    method: str = "RK45"
    rtol: float = 1e-12
    atol: float = 1e-12
    max_step: float = float("inf")
    cycles: int = 25
    output_per_cycle: int = 100

SOLVER = SolverSettings()
N = 20
MASS = 1.0
DAMPING = 0.02
CONTACT = 1.2e4
M0 = N * MASS / 2
IMPACT_MASS = 0.01 * M0
IMPACT_VELOCITY = 0.25

SOURCE_MANIFEST = {
    "presentation": "Week 6 Presentation.pdf; pages 3–10; page 11 is optional future work",
    "simulation_basis": "main_disorder_v2 Python function pasted in this conversation; original MATLAB force equations",
    "uploaded_zip_inspected_during_assembly": False,
    "source_main_grid": "arange(floor(T/dt)+1)*dt; both solves end at the final output time",
    "sensitivity": "Multiplicative -50% to +50%; presentation baseline; no feasibility filtering",
    "joint_gp": "Squared-slack parameterization; positive-energy and category bounds enforced",
    "saved_results": "Earlier GP logs; not generated by running this notebook",
    "native_matlab_executed_by_notebook": False,
    "fresh_search_matches_historical_values_guaranteed": False,
    "scipy_documentation": "https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html",
    "gp_documentation": "https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html",
    "sobol_documentation": "https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.qmc.Sobol.html",
}
ENVIRONMENT = {"python": sys.version, "numpy": np.__version__, "scipy": scipy.__version__,
               "scikit_learn": sklearn.__version__, "run_mode": RUN_MODE,
               "solver": asdict(SOLVER), "source": SOURCE_MANIFEST}
(OUTPUT_ROOT / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2))
display(pd.DataFrame([
    {"setting": "Run mode", "value": RUN_MODE},
    {"setting": "Fresh sensitivity simulations", "value": RUN_SENSITIVITY},
    {"setting": "Fresh final-candidate DEM checks", "value": RECHECK_SELECTED_DESIGNS},
    {"setting": "Optional next-step GP", "value": RUN_NEXT_STEP},
    {"setting": "Output", "value": str(OUTPUT_ROOT)},
]))

## 2. Python DEM and RK45

In [ ]:
@njit(cache=False)
def impact_force(C, x1, x2):
    if x1 > x2:
        return C * (x1 - x2) ** 1.5
    return 0.0

@njit(cache=False)
def impact_equation_of_motion_hetero(t, x, nonlinear_spring_info, material_info, F):
    number_of_masses = x.size // 2
    dx = np.zeros_like(x)
    m, m_impact, C_impact, zeta = material_info
    dx[0] = x[1]
    dx[1] = (F - impact_force(C_impact, x[0], x[2])) / m_impact
    for i in range(1, number_of_masses):
        dx[2 * i] = x[2 * i + 1]
        q_right = 0.0 if i == number_of_masses - 1 else x[2 * (i + 1)]
        v_right = 0.0 if i == number_of_masses - 1 else x[2 * (i + 1) + 1]
        delta_right = q_right - x[2 * i]
        c1, c2, c3 = nonlinear_spring_info[i - 1]
        right_force = c1 * delta_right
        if delta_right < 0:
            right_force -= c2 * delta_right ** 2
            right_force += c3 * delta_right ** 3
        if i == 1:
            left_force = impact_force(C_impact, x[0], x[2])
            damping_force = 2 * zeta * (v_right - x[3])
        else:
            delta_left = x[2 * i] - x[2 * (i - 1)]
            c1, c2, c3 = nonlinear_spring_info[i - 2]
            left_force = -c1 * delta_left
            if delta_left < 0:
                left_force += c2 * delta_left ** 2
                left_force -= c3 * delta_left ** 3
            damping_force = 2 * zeta * (v_right - 2 * x[2 * i + 1] + x[2 * (i - 1) + 1])
        dx[2 * i + 1] = (right_force + left_force + damping_force) / m
    return dx


def coefficient_arrays(c2, c3):
    c2 = np.asarray(c2, dtype=float).reshape(-1)
    c3 = np.asarray(c3, dtype=float).reshape(-1)
    if c2.shape != (4,) or c3.shape != (4,) or not np.isfinite(np.r_[c2, c3]).all():
        raise ValueError("Supply four finite c2 values and four finite c3 values.")
    return c2, c3


def simulation_grid(c2, c3, settings=SOLVER):
    c2, c3 = coefficient_arrays(c2, c3)
    k_f = float(np.max(np.r_[1.0, 1 + 2 * c2 + 3 * c3]))
    f0 = np.sqrt(1 / M0) / (2 * np.pi)
    f = np.sqrt(k_f / M0) / (2 * np.pi)
    dt = (1 / f) / settings.output_per_cycle
    T = (1 / f0) * settings.cycles
    t_eval = np.arange(int(np.floor(T / dt)) + 1) * dt
    if t_eval.size < 2:
        raise ValueError("Insufficient simulation output times.")
    return t_eval, T


def solve_pair(c2, c3, settings=SOLVER, keep_last_series=False):
    c2, c3 = coefficient_arrays(c2, c3)
    t_eval, requested_T = simulation_grid(c2, c3, settings)
    material_info = np.array([MASS, IMPACT_MASS, CONTACT, DAMPING], dtype=float)
    initial_values = np.zeros(2 * (N + 1))
    initial_values[1] = IMPACT_VELOCITY
    C1 = np.ones(N)
    C2, C3 = np.repeat(c2, 5), np.repeat(c3, 5)
    peaks, peak_times, counts, series = [], [], [], []
    started = time.perf_counter()
    for cc2, cc3 in ((np.zeros(N), np.zeros(N)), (C2, C3)):
        spring_info = np.column_stack((C1, cc2, cc3))
        result = solve_ivp(
            impact_equation_of_motion_hetero, (0.0, float(t_eval[-1])),
            initial_values, method=settings.method, t_eval=t_eval,
            args=(spring_info, material_info, 0.0),
            rtol=settings.rtol, atol=np.full(2 * (N + 1), settings.atol),
            max_step=settings.max_step,
        )
        if not result.success or result.t.size != t_eval.size:
            raise RuntimeError("Incomplete DEM solve: " + result.message)
        if not np.isfinite(result.y).all():
            raise RuntimeError("Nonfinite DEM state.")
        last_ke = 0.5 * result.y[2 * N + 1] ** 2
        index = int(np.argmax(last_ke))
        peaks.append(float(last_ke[index]))
        peak_times.append(float(t_eval[index]))
        counts.append(int(result.nfev))
        if keep_last_series:
            series.append(last_ke.copy())
        del result
    if peaks[0] <= 0 or peaks[1] <= 0:
        raise RuntimeError("Nonpositive last-mass energy.")
    ratio = peaks[1] / peaks[0]
    answer = {
        "ratio": ratio, "KE_ratio": ratio, "log10_KE_ratio": float(np.log10(ratio)),
        "linear_peak_ke": peaks[0], "nonlinear_peak_ke": peaks[1],
        "linear_peak_time": peak_times[0], "nonlinear_peak_time": peak_times[1],
        "nfev_linear": counts[0], "nfev_nonlinear": counts[1],
        "time_end": float(t_eval[-1]), "requested_T": float(requested_T),
        "output_points": int(t_eval.size), "seconds": time.perf_counter() - started,
        "method": settings.method, "rtol": settings.rtol, "atol": settings.atol,
        "max_step": settings.max_step, "status": "ok", "source": "fresh_python_dem",
    }
    if keep_last_series:
        answer.update(time=t_eval, linear_last_ke=series[0], nonlinear_last_ke=series[1])
    return answer


def main_disorder_v2(c2, c3):
    return solve_pair(c2, c3)["ratio"]


EXTERNAL_DEM = None
if DEM_ZIP_PATH:
    external_dir = OUTPUT_ROOT / "uploaded_dem"
    external_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(DEM_ZIP_PATH) as archive:
        for name in archive.namelist():
            if name.endswith(".py") and "__MACOSX" not in name:
                (external_dir / Path(name).name).write_bytes(archive.read(name))
    sys.path.insert(0, str(external_dir))
    spec = importlib.util.spec_from_file_location("uploaded_main", external_dir / "main_disorder_v2.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    EXTERNAL_DEM = module.main_disorder_v2


def run_dem(c2, c3, settings=SOLVER):
    if EXTERNAL_DEM is not None and settings == SOLVER:
        start = time.perf_counter()
        ratio = float(EXTERNAL_DEM(np.asarray(c2), np.asarray(c3)))
        if not np.isfinite(ratio) or ratio <= 0:
            raise RuntimeError("The uploaded simulator returned an invalid KE ratio.")
        return {"ratio": ratio, "KE_ratio": ratio, "log10_KE_ratio": float(np.log10(ratio)),
                "status": "ok", "source": "uploaded_dem_python", "seconds": time.perf_counter() - start,
                "method": "uploaded_main_disorder_v2"}
    return solve_pair(c2, c3, settings)


def validate_physics():
    rng = np.random.default_rng(901)
    errors = []
    for _ in range(100):
        x = rng.normal(0, 0.02, 42)
        c2 = rng.uniform(-120, 0, 20)
        c3 = rng.uniform(0, 4000, 20)
        info = np.column_stack((np.ones(20), c2, c3))
        material = np.array([MASS, IMPACT_MASS, CONTACT, DAMPING])
        dx = impact_equation_of_motion_hetero(0., x, info, material, 0.)
        v = x[1::2]
        power = IMPACT_MASS * v[0] * dx[1] + np.dot(v[1:], dx[3::2])
        power += CONTACT * max(x[0] - x[2], 0.) ** 1.5 * (v[0] - v[1])
        dissipated = 0.
        for j in range(20):
            q_right = 0. if j == 19 else x[2 * (j + 2)]
            v_right = 0. if j == 19 else v[j + 2]
            s = x[2 * (j + 1)] - q_right
            dv = v[j + 1] - v_right
            force = s + (c2[j] * s ** 2 + c3[j] * s ** 3 if s > 0 else 0.)
            power += force * dv
            dissipated += 2 * DAMPING * dv ** 2
        errors.append(abs(power + dissipated))
    assert max(errors) < 1e-9
    return {"power_balance_tests": len(errors), "maximum_absolute_error": max(errors)}

PHYSICS_CHECKS = validate_physics()
display(pd.DataFrame([PHYSICS_CHECKS]))

## 3. Physical Constraints and Simulation Checkpoints

In [ ]:
def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.ndarray):
        return clean_json(value.tolist())
    if isinstance(value, (np.integer, np.floating)):
        return clean_json(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    return value


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(clean_json(value), indent=2, allow_nan=False))
    temporary.replace(path)


def read_rows(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines):
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i != len(lines) - 1:
                raise
            path.write_text("\n".join(json.dumps(r) for r in rows) + "\n")
    return rows


def flat_frame(rows):
    output = []
    for row in rows:
        entry = {k: v for k, v in row.items() if not isinstance(v, (list, dict))}
        for key in ("c2", "c3"):
            for family, value in zip("ABCD", row.get(key, [])):
                entry[f"{key}_{family}"] = value
        for i, value in enumerate(row.get("z", [])):
            entry[f"z_{i}"] = value
        if "ratio" in row:
            entry["KE_ratio"] = row["ratio"]
            entry["KE_percent"] = 100 * row["ratio"]
            entry["log10_KE_ratio"] = np.log10(row["ratio"])
        output.append(entry)
    return pd.DataFrame(output)


def dimension(category):
    if category not in PAPER_RATIOS:
        raise ValueError(category)
    return 2 if category == "homogeneous" else 8


def design_valid(c2, c3, category="heterogeneous"):
    c2, c3 = coefficient_arrays(c2, c3)
    bounds = np.all((-120 <= c2) & (c2 <= 0)) and np.all((0 <= c3) & (c3 <= 4000))
    energy = np.all(c3 > (2 / 9) * c2 ** 2)
    nonbistable = category != "nonbistable" or np.all(c2 ** 2 <= 4 * c3)
    homogeneous = category != "homogeneous" or (np.all(c2 == c2[0]) and np.all(c3 == c3[0]))
    return bool(bounds and energy and nonbistable and homogeneous)


def decode(z, category, power=2):
    z = np.asarray(z, dtype=float).reshape(-1)
    if len(z) != dimension(category) or not np.isfinite(z).all() or np.any((z < 0) | (z > 1)):
        raise ValueError("Invalid normalized design coordinates.")
    k = len(z) // 2
    c2 = -120 * z[:k]
    a = 0.25 if category == "nonbistable" else 2 / 9
    lower = a * c2 ** 2 + 1e-7
    c3 = lower + (4000 - lower) * z[k:] ** power
    if k == 1:
        c2, c3 = np.repeat(c2, 4), np.repeat(c3, 4)
    if not design_valid(c2, c3, category):
        raise ValueError("Decoded coefficients are outside the constrained design space.")
    return c2, c3


def encode(c2, c3, category, power=2):
    c2, c3 = coefficient_arrays(c2, c3)
    if not design_valid(c2, c3, category):
        raise ValueError("Cannot encode an inadmissible design.")
    if category == "homogeneous":
        c2, c3 = c2[:1], c3[:1]
    a = 0.25 if category == "nonbistable" else 2 / 9
    lower = a * c2 ** 2 + 1e-7
    slack = (c3 - lower) / (4000 - lower)
    if np.min(slack) < -1e-10:
        raise ValueError("The design lies outside the interior-margin parameterization.")
    return np.r_[-c2 / 120, np.clip(slack, 0, 1) ** (1 / power)]


def sobol_designs(n, d, seed):
    if n < 1 or n & (n - 1):
        raise ValueError("Use a power of two for Sobol initialization.")
    return qmc.Sobol(d=d, scramble=True, seed=seed).random_base2(int(np.log2(n)))


class DesignLog:
    def __init__(self, directory, category, config):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.category = category
        self.path = self.directory / "evaluations.jsonl"
        self.config = clean_json({"category": category, "solver": asdict(SOLVER),
                                  "source": SOURCE_MANIFEST["simulation_basis"],
                                  "external_source": DEM_ZIP_PATH, **config})
        manifest = self.directory / "configuration.json"
        if manifest.exists() and json.loads(manifest.read_text()) != self.config:
            raise ValueError("Configuration changed. Choose a different output directory.")
        write_json(manifest, self.config)
        self.rows = read_rows(self.path)
        self.index = {r["evaluation_id"]: r for r in self.rows}

    def evaluate(self, z, evaluation_id, phase, region=None, predicted_log_ratio=None, power=2):
        z = np.asarray(z, dtype=float)
        c2, c3 = decode(z, self.category, power)
        return self.evaluate_coefficients(c2, c3, evaluation_id, phase, region,
                                         predicted_log_ratio, z.tolist(), True)

    def evaluate_coefficients(self, c2, c3, evaluation_id, phase, region=None,
                              predicted_log_ratio=None, z=None, enforce=True):
        c2, c3 = coefficient_arrays(c2, c3)
        if evaluation_id in self.index:
            old = self.index[evaluation_id]
            if not np.array_equal(np.r_[old["c2"], old["c3"]], np.r_[c2, c3]):
                raise RuntimeError("Checkpoint design mismatch.")
            return old
        admissible = design_valid(c2, c3, self.category)
        if enforce and not admissible:
            raise ValueError("Inadmissible search proposal.")
        try:
            result = run_dem(c2, c3)
        except Exception as error:
            failure = {"evaluation_id": evaluation_id, "c2": c2.tolist(), "c3": c3.tolist(),
                       "phase": phase, "error": repr(error), "status": "failed"}
            with (self.directory / "failures.jsonl").open("a") as file:
                file.write(json.dumps(failure) + "\n")
            raise
        row = clean_json({"evaluation": len(self.rows) + 1, "evaluation_id": evaluation_id,
                          "category": self.category, "phase": phase, "region": region,
                          "c2": c2.tolist(), "c3": c3.tolist(), "admissible": admissible,
                          "predicted_log_ratio": predicted_log_ratio, **result})
        if z is not None:
            row["z"] = z
        with self.path.open("a") as file:
            file.write(json.dumps(row, allow_nan=False) + "\n")
            file.flush()
        self.rows.append(row)
        self.index[evaluation_id] = row
        if row["ratio"] <= min(r["ratio"] for r in self.rows):
            write_json(self.directory / "best.json", row)
        return row

    def export(self):
        flat_frame(self.rows).to_csv(self.directory / "dataset.csv", index=False, float_format="%.17g")

## 4. One-Parameter Sensitivity Functions

In [ ]:
def sensitivity_experiment(directory):
    log = DesignLog(directory, "heterogeneous", {
        "stage": "presentation_sensitivity", "base_c2": BASE_C2.tolist(),
        "base_c3": BASE_C3.tolist(), "perturbations": PERTURBATIONS.tolist(),
        "enforce_constraints": False,
    })
    rows = []
    for coefficient in ("c2", "c3"):
        for family in range(4):
            for j, change in enumerate(PERTURBATIONS):
                c2, c3 = BASE_C2.copy(), BASE_C3.copy()
                if coefficient == "c2":
                    c2[family] *= 1 + change
                else:
                    c3[family] *= 1 + change
                row = log.evaluate_coefficients(c2, c3, f"{coefficient}_{family}_{j}",
                                                "one_at_a_time", enforce=False).copy()
                row.update(coefficient=coefficient, family=family, perturbation_percent=100 * float(change),
                           parameter_value=float(c2[family] if coefficient == "c2" else c3[family]))
                rows.append(row)
            print(f"Sensitivity: {coefficient}_{family + 1}", flush=True)
    frame = flat_frame(rows)
    frame.to_csv(Path(directory) / "sensitivity.csv", index=False, float_format="%.17g")
    return rows


def finish_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES / (filename + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES / (filename + ".svg"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_sensitivity(rows):
    data = pd.DataFrame(rows)
    for coefficient in ("c2", "c3"):
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for family in range(4):
            part = data[(data.coefficient == coefficient) & (data.family == family)].sort_values("perturbation_percent")
            ax.plot(part.perturbation_percent, part.ratio, marker="o", ms=3,
                    label=f"Family {family + 1}")
        ax.axvline(0, linestyle=":", linewidth=1)
        ax.set(xlabel="Multiplicative perturbation from baseline (%)", ylabel="Peak KE ratio (decimal)",
               title=f"One-at-a-time sensitivity: {coefficient}")
        ax.legend()
        ax.grid(alpha=0.18)
        finish_figure(fig, f"sensitivity_{coefficient}_families")
    for (coefficient, family), part in data.groupby(["coefficient", "family"]):
        part = part.sort_values("parameter_value")
        baseline = BASE_C2[family] if coefficient == "c2" else BASE_C3[family]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(part.parameter_value, part.ratio, marker="o", ms=3)
        ax.axvline(baseline, linestyle="--", linewidth=1, label=f"Baseline = {baseline:g}")
        ax.set(xlabel=f"{coefficient}, family {family + 1}", ylabel="Peak KE ratio (decimal)",
               title=f"{coefficient}, family {family + 1}: other coefficients fixed")
        ax.legend()
        ax.grid(alpha=0.18)
        finish_figure(fig, f"sensitivity_{coefficient}_{family + 1}")

## 5. Gaussian Process Training

In [ ]:
def hyper_optimizer(objective, initial_theta, bounds):
    result = minimize(objective, initial_theta, jac=True, bounds=bounds, method="L-BFGS-B",
                      options={"maxiter": 40, "ftol": 1e-8})
    return result.x, result.fun


def fit_gp(X, y, kernel, alpha, optimize=True, maxiter=40):
    def optimizer(objective, initial_theta, bounds):
        result = minimize(objective, initial_theta, jac=True, bounds=bounds, method="L-BFGS-B",
                          options={"maxiter": maxiter, "ftol": 1e-8})
        return result.x, result.fun
    model = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True,
                                     optimizer=optimizer if optimize else None,
                                     n_restarts_optimizer=0, random_state=2026)
    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
        warnings.simplefilter("always")
        model.fit(np.asarray(X), np.asarray(y))
    if caught:
        with (OUTPUT_ROOT / "gp_fit_warnings.log").open("a") as file:
            for warning in caught:
                file.write(str(warning.message) + "\n")
    return model


def predict_gp(model, X):
    with threadpool_limits(limits=1):
        mean, std = model.predict(np.asarray(X), return_std=True)
    return mean, np.maximum(std, 1e-12)


def expected_improvement(mean, std, best, margin=0.):
    gain = best - mean - margin
    standardized = gain / np.maximum(std, 1e-12)
    return gain * norm.cdf(standardized) + std * norm.pdf(standardized)


def gp_mean_gradient(model, x):
    x = np.asarray(x)
    length = np.asarray(model.kernel_.k2.length_scale)
    amplitude = model.kernel_.k1.constant_value
    delta = x - model.X_train_
    distance = np.sqrt(np.sum((delta / length) ** 2, axis=1))
    exponential = np.exp(-np.sqrt(5) * distance)
    covariance = amplitude * (1 + np.sqrt(5) * distance + (5 / 3) * distance ** 2) * exponential
    derivative = (-amplitude * (5 / 3) * (1 + np.sqrt(5) * distance)[:, None]
                  * exponential[:, None] * delta / length ** 2)
    mean = float(np.asarray(model._y_train_mean).item() +
                 np.asarray(model._y_train_std).item() * (covariance @ model.alpha_))
    gradient = np.asarray(model._y_train_std).item() * derivative.T @ model.alpha_
    return mean, gradient

## 6. Initial Sobol Designs and First GP Search

In [ ]:
def first_gp_search(category, seed, rounds, output_dir):
    log = DesignLog(output_dir, category, {"stage": "initial_gp", "seed": seed,
                    "initial": INITIAL_DESIGNS, "rounds": rounds, "batch": FIRST_GP_BATCH,
                    "mapping": "squared_slack", "kernel": "Matern_3_2"})
    checkpoint_path = log.directory / "checkpoint.json"
    dimension_value = dimension(category)
    rng = np.random.default_rng(seed)
    initial = sobol_designs(INITIAL_DESIGNS, dimension_value, seed)
    X, y = [], []
    for j, z in enumerate(initial):
        row = log.evaluate(z, f"initial_{j}", "initial_sobol")
        X.append(z.copy())
        y.append(np.log10(row["ratio"]))
    X, y = np.array(X), np.array(y)
    length = 0.15
    kernel = (ConstantKernel(1., (0.1, 10.)) *
              Matern(np.full(dimension_value, 0.15), (0.004, 2.), nu=1.5) +
              WhiteKernel(1e-5, (1e-8, 0.02)))
    completed = 0
    if checkpoint_path.exists():
        state = json.loads(checkpoint_path.read_text())
        X, y = np.array(state["X"]), np.array(state["y"])
        completed, length = state["completed_rounds"], state["length"]
        rng.bit_generator.state = state["rng_state"]
        if state["kernel_theta"] is not None:
            kernel = kernel.clone_with_theta(np.array(state["kernel_theta"]))
    last_model = None
    for iteration in range(completed, rounds):
        center = X[np.argmin(y)]
        if len(X) <= 384:
            ids = np.arange(len(X))
        else:
            ids = np.unique(np.r_[np.argsort(y)[:128],
                                  np.argsort(np.sum((X - center) ** 2, axis=1))[:128],
                                  rng.choice(len(X), 128, replace=False)])
        model = fit_gp(X[ids], y[ids], kernel, alpha=1e-9, optimize=iteration % 5 == 0)
        kernel = model.kernel_
        elite = X[np.argsort(y)[:min(8, len(X))]]
        centers = elite[rng.integers(len(elite), size=4096)]
        scales = 10 ** rng.uniform(-3.2, np.log10(length), size=(4096, 1))
        local = np.clip(centers + rng.normal(size=(4096, dimension_value)) * scales, 0, 1)
        pool = np.vstack([rng.random((2048, dimension_value)), local])
        pool = pool[np.min(cdist(pool, X), axis=1) > 1e-6]
        mean, std = predict_gp(model, pool)
        ei = expected_improvement(mean, std, float(y.min()), margin=0.002)
        indices = []
        per_score = max(1, (FIRST_GP_BATCH - 2) // 2)
        for order in (np.argsort(-ei), np.argsort(mean)):
            added = 0
            for j in order:
                if indices and np.min(np.linalg.norm(pool[indices] - pool[j], axis=1)) < 0.005:
                    continue
                indices.append(int(j))
                added += 1
                if added == per_score:
                    break
        proposals = [pool[j] for j in indices[:FIRST_GP_BATCH]]
        predictions = [float(mean[j]) for j in indices[:FIRST_GP_BATCH]]
        while len(proposals) < FIRST_GP_BATCH:
            proposals.append(rng.random(dimension_value))
            predictions.append(None)
        previous = float(y.min())
        for j, (z, prediction) in enumerate(zip(proposals, predictions)):
            row = log.evaluate(z, f"round_{iteration}_{j}", f"gp_round_{iteration + 1}",
                               predicted_log_ratio=prediction)
            X = np.vstack([X, z])
            y = np.r_[y, np.log10(row["ratio"])]
        length = min(0.3, length * 1.12) if y.min() < previous - 0.01 else max(0.01, length * 0.93)
        write_json(checkpoint_path, {"completed_rounds": iteration + 1, "X": X, "y": y,
                    "length": length, "kernel_theta": kernel.theta, "rng_state": rng.bit_generator.state})
        last_model = model
        print(f"{category}, seed {seed}, round {iteration + 1}/{rounds}, best = {100 * 10 ** y.min():.6f}%", flush=True)
    if last_model is not None:
        last_model.optimizer = None
        joblib.dump(last_model, log.directory / "last_search_gp.joblib")
    log.export()
    write_json(log.directory / "summary.json", {"category": category, "evaluations": len(log.rows),
               "best": min(log.rows, key=lambda row: row["ratio"]), "source": "new_rk45_search"})
    return log.rows

## 7. Six Regions and Three-Region Continuation

In [ ]:
def choose_seeds(archive, count=N_REGIONS, separation=REGION_SEPARATION):
    seeds = []
    for row in sorted(archive, key=lambda row: row["ratio"]):
        z = np.array(row["z"])
        if not seeds or min(np.linalg.norm(z - np.array(other["z"])) for other in seeds) > separation:
            seeds.append(row)
        if len(seeds) == count:
            return seeds
    raise RuntimeError(f"Only {len(seeds)} separated seeds found. Expand the initial archive; do not substitute untested seeds.")


def restore_regions(raw):
    states = []
    for item in raw:
        state = dict(item)
        state["X"] = np.array(state["X"], dtype=float)
        state["y"] = np.array(state["y"], dtype=float)
        states.append(state)
    return states


def propose_local(model, center, radius, X, y, rng, batch=7):
    dimension_value = len(center)
    bounds = list(zip(np.maximum(-1., -center / radius),
                      np.minimum(1., (1 - center) / radius)))
    proposals = []
    for j in range(4):
        x0 = np.zeros(dimension_value) if j == 0 else rng.uniform(-0.6, 0.6, dimension_value)
        x0 = np.clip(x0, np.array(bounds)[:, 0], np.array(bounds)[:, 1])
        result = minimize(lambda value: gp_mean_gradient(model, value), x0, jac=True,
                          method="L-BFGS-B", bounds=bounds,
                          options={"maxiter": 60, "ftol": 1e-10})
        z = np.clip(center + radius * result.x, 0, 1)
        if np.min(np.linalg.norm(X - z, axis=1)) <= 1e-7:
            continue
        if not proposals or min(np.linalg.norm(other - z) for other in proposals) > 1e-6:
            proposals.append(z)
    pool = np.clip(center + rng.normal(size=(1600, dimension_value)) * radius * 0.45, 0, 1)
    mean, std = predict_gp(model, (pool - center) / radius)
    ei = expected_improvement(mean, std, float(y.min()))
    for j in np.argsort(-ei):
        if np.min(np.linalg.norm(X - pool[j], axis=1)) <= 1e-9:
            continue
        if not proposals or min(np.linalg.norm(other - pool[j]) for other in proposals) > 0.02 * radius:
            proposals.append(pool[j])
        if len(proposals) >= batch - 1:
            break
    proposals = proposals[:batch - 1]
    while len(proposals) < batch:
        z = np.clip(center + rng.normal(size=dimension_value) * radius * 0.4, 0, 1)
        if np.min(np.linalg.norm(X - z, axis=1)) > 1e-9 and (not proposals or
                min(np.linalg.norm(other - z) for other in proposals) > 1e-9):
            proposals.append(z)
    return proposals


def multiregion_search(category, archive, output_dir, rounds=REGION_ROUNDS, resume_regions=None):
    log = DesignLog(output_dir, category, {"stage": "six_regions" if resume_regions is None else "three_region_continuation",
                    "rounds": rounds, "neighbors": REGION_NEIGHBORS, "regions": N_REGIONS,
                    "batch": REGION_BATCH, "distance": REGION_SEPARATION, "mapping": "squared_slack"})
    checkpoint_path = log.directory / "checkpoint.json"
    rng = np.random.default_rng(928341)
    completed = 0
    if checkpoint_path.exists():
        checkpoint = json.loads(checkpoint_path.read_text())
        states = restore_regions(checkpoint["states"])
        completed = checkpoint["completed_rounds"]
        rng.bit_generator.state = checkpoint["rng_state"]
    elif resume_regions is not None:
        states = restore_regions(sorted(resume_regions, key=lambda state: min(state["y"]))[:CONTINUE_REGIONS])
        for state in states:
            state["success"] = 0
            state["fail"] = 0
        write_json(log.directory / "provenance.json", {"continued_original_regions": [s["region"] for s in states],
                   "inherited_local_records": sum(len(s["y"]) for s in states),
                   "inherited_archive_rows": len(archive)})
    else:
        seeds = choose_seeds(archive)
        write_json(log.directory / "provenance.json", {"archive_rows": len(archive),
                   "separation": REGION_SEPARATION, "seeds": seeds, "published_optimum_seeds": False})
        states = []
        for region, seed in enumerate(seeds):
            center = np.array(seed["z"])
            samples = [center] + [np.clip(center + rng.normal(size=len(center)) * REGION_INITIAL_SD, 0, 1)
                                  for _ in range(REGION_NEIGHBORS)]
            labels = []
            for j, z in enumerate(samples):
                row = log.evaluate(z, f"init_{region}_{j}", "seed_recheck" if j == 0 else "local_initialization", region)
                labels.append(np.log10(row["ratio"]))
            states.append({"region": region, "X": np.array(samples), "y": np.array(labels),
                           "radius": REGION_INITIAL_RADIUS, "success": 0, "fail": 0,
                           "original_center": center.tolist()})
        write_json(checkpoint_path, {"states": states, "completed_rounds": 0, "rng_state": rng.bit_generator.state})
    for iteration in range(completed, rounds):
        for slot, state in enumerate(states):
            X, y, radius = state["X"], state["y"], state["radius"]
            center = X[np.argmin(y)]
            ids = np.argsort(np.linalg.norm(X - center, axis=1))[:160]
            kernel = ConstantKernel(1., (0.01, 100.)) * Matern(np.ones(len(center)), (0.01, 30.), nu=2.5)
            model = fit_gp((X[ids] - center) / radius, y[ids], kernel, alpha=1e-5, maxiter=25)
            proposals = propose_local(model, center, radius, X, y, rng, REGION_BATCH)
            previous = float(y.min())
            for j, z in enumerate(proposals):
                predicted = float(model.predict(((z - center) / radius)[None])[0])
                row = log.evaluate(z, f"round_{iteration}_{slot}_{j}", f"gp_round_{iteration + 1}",
                                   state["region"], predicted)
                X = np.vstack([X, z])
                y = np.r_[y, np.log10(row["ratio"])]
            state["X"], state["y"] = X, y
            if y.min() < previous - 1e-4:
                state["success"] += 1
                state["fail"] = 0
                if state["success"] >= 2:
                    state["radius"] = min(0.2, radius * 1.4)
                    state["success"] = 0
            else:
                state["fail"] += 1
                state["success"] = 0
                if state["fail"] >= 3:
                    state["radius"] = max(0.0002, radius * 0.55)
                    state["fail"] = 0
            model.optimizer = None
            joblib.dump({"gp": model, "center": center, "radius": radius, "category": category},
                        log.directory / f"region_{state['region']}_model.joblib")
        write_json(checkpoint_path, {"states": states, "completed_rounds": iteration + 1,
                                    "rng_state": rng.bit_generator.state})
        print(f"{category}, {len(states)} regions, round {iteration + 1}/{rounds}, best = "
              f"{100 * min(10 ** np.min(s['y']) for s in states):.6f}%", flush=True)
    log.export()
    write_json(log.directory / "summary.json", {"evaluations": len(log.rows),
               "best": min(log.rows, key=lambda row: row["ratio"]) if log.rows else None,
               "regions": [s["region"] for s in states]})
    return log.rows, states

## 8. Search Pipeline

In [ ]:
def load_presentation_records(directory):
    directory = Path(directory)
    source_records = {}
    for category in PAPER_RATIOS:
        paths = sorted((directory / "results").glob(f"{category}_gp_seed*/evaluations.jsonl"))
        paths += sorted((directory / "results/extended").glob(f"{category}_gp_seed*/evaluations.jsonl"))
        rows = []
        for path in paths:
            for row in read_rows(path):
                row.update(category=category, source="saved_presentation_archive", source_file=str(path.relative_to(directory)))
                rows.append(row)
        source_records[category] = rows
    local_records = {category: read_rows(directory / f"improvement/multiregion_gp/{category}/evaluations.jsonl")
                     for category in ("heterogeneous", "nonbistable")}
    continuation = read_rows(directory / "improvement/gp_resume/heterogeneous/evaluations.jsonl")
    candidates = [r for r in json.loads((directory / "IMPROVED_CANDIDATES.json").read_text())
                  if r["selection"] == "ml_only"]
    for row in candidates:
        row["source"] = "saved_presentation_verified_candidate"
    return source_records, local_records, continuation, candidates


def run_new_pipeline():
    archives, local_records, region_states, candidates = {}, {}, {}, []
    continuation = []
    for category in CATEGORIES:
        rows = []
        for seed in FIRST_GP_SEEDS:
            rows += first_gp_search(category, seed, FIRST_GP_ROUNDS,
                     OUTPUT_ROOT / "new_search" / f"{category}_gp_seed{seed}")
        if EXTENDED_ROUNDS > FIRST_GP_ROUNDS:
            rows += first_gp_search(category, FIRST_GP_SEEDS[0], EXTENDED_ROUNDS,
                     OUTPUT_ROOT / "new_search/extended" / f"{category}_gp_seed{FIRST_GP_SEEDS[0]}")
        archives[category] = rows
        if category == "homogeneous":
            best = dict(min(rows, key=lambda row: row["ratio"]))
        else:
            stage, states = multiregion_search(category, rows,
                            OUTPUT_ROOT / "new_search/multiregion" / category)
            local_records[category], region_states[category] = stage, states
            best = dict(min(stage, key=lambda row: row["ratio"]))
            if category == "heterogeneous":
                continuation, continued_states = multiregion_search(
                    category, rows, OUTPUT_ROOT / "new_search/continuation" / category,
                    rounds=CONTINUE_ROUNDS, resume_regions=states)
                best = dict(min(stage + continuation, key=lambda row: row["ratio"]))
        best.update(category=category, selection="gp_only", source="new_rk45_search",
                    screening_ratio=best["ratio"])
        candidates.append(best)
    write_json(OUTPUT_ROOT / "new_search/selected_candidates.json", candidates)
    return archives, local_records, continuation, candidates

## 9. Sensitivity Data and Figures

In [ ]:
if RUN_SENSITIVITY:
    sensitivity_rows = sensitivity_experiment(OUTPUT_ROOT / "sensitivity")
else:
    sensitivity_rows = []

if sensitivity_rows:
    baseline_rows = [row for row in sensitivity_rows if abs(row["perturbation_percent"]) < 1e-8]
    display(pd.DataFrame([{
        "baseline_KE_ratio": baseline_rows[0]["ratio"],
        "baseline_KE_percent": 100 * baseline_rows[0]["ratio"],
        "presentation_rounded_ratio": 0.1481,
        "positive_energy_constraint_satisfied": design_valid(BASE_C2, BASE_C3),
        "sensitivity_rows": len(sensitivity_rows),
        "result_source": baseline_rows[0]["source"],
    }]))
    plot_sensitivity(sensitivity_rows)
else:
    print("Sensitivity experiment skipped. Set RUN_SENSITIVITY=True to generate it.")


## 10. Initial Search and Regional Optimization

In [ ]:
if RUN_MODE == "saved_presentation":
    archives, local_records, continuation_rows, selected_candidates = load_presentation_records(PRESENTATION_DATA)
else:
    archives, local_records, continuation_rows, selected_candidates = run_new_pipeline()

archive_counts = []
for category, rows in archives.items():
    unique = {tuple(np.r_[row["c2"], row["c3"]]) for row in rows}
    archive_counts.append({"category": category, "archive_rows": len(rows),
                           "unique_designs": len(unique), "repeated_records": len(rows) - len(unique),
                           "source": "saved_presentation_archive" if RUN_MODE == "saved_presentation" else "new_rk45_search"})
    flat_frame(rows).to_csv(OUTPUT_ROOT / f"initial_archive_{category}.csv", index=False, float_format="%.17g")
display(pd.DataFrame(archive_counts))

seed_selection = []
for category in ("heterogeneous", "nonbistable"):
    if category not in archives:
        continue
    seeds = choose_seeds(archives[category])
    for region, row in enumerate(seeds):
        seed_selection.append({"category": category, "region": region,
                               "archived_KE_percent": 100 * row["ratio"],
                               "c2": row["c2"], "c3": row["c3"], "z": row["z"]})
write_json(OUTPUT_ROOT / "selected_region_seeds.json", seed_selection)
display(pd.DataFrame(seed_selection)[["category", "region", "archived_KE_percent", "c2", "c3"]])

## 11. Local GP Refit — Training Demonstration

In [ ]:
final_models = {}
for candidate in selected_candidates:
    category = candidate["category"]
    center = encode(candidate["c2"], candidate["c3"], category)
    rows = list(archives[category]) + list(local_records.get(category, []))
    if category == "heterogeneous":
        rows += continuation_rows
    unique_rows = {}
    for row in rows:
        unique_rows[tuple(row["z"])] = row
    rows = list(unique_rows.values())
    X = np.array([row["z"] for row in rows])
    y = np.log10([row["ratio"] for row in rows])
    ids = np.argsort(np.linalg.norm(X - center, axis=1))[:160]
    radius = 0.07
    kernel = ConstantKernel(1., (0.01, 100.)) * Matern(np.ones(len(center)), (0.01, 30.), nu=2.5)
    model = fit_gp((X[ids] - center) / radius, y[ids], kernel, 1e-5)
    model.optimizer = None
    final_models[category] = {"gp": model, "center": center, "radius": radius,
                               "category": category, "purpose": "refit demonstration; not a new search result"}
    joblib.dump(final_models[category], OUTPUT_ROOT / f"{category}_local_gp_refit.joblib")
    mean, std = predict_gp(model, np.zeros((1, len(center))))
    display(pd.DataFrame([{"category": category, "training_rows": len(ids),
                           "query": "selected existing design; not held out",
                           "GP_central_prediction_percent": 100 * 10 ** mean[0],
                           "GP_log10_standard_deviation": std[0],
                           "kernel": str(model.kernel_)}]))

## 12. DEM Verification and KE Ratios

In [ ]:
def validate_candidates(candidates):
    rows = []
    for candidate in candidates:
        category = candidate["category"]
        c2, c3 = np.array(candidate["c2"]), np.array(candidate["c3"])
        if not design_valid(c2, c3, category):
            raise ValueError("Selected design violates its constraints.")
        for method in ("RK45", "DOP853"):
            settings = SolverSettings(method=method, rtol=1e-12, atol=1e-12)
            result = solve_pair(c2, c3, settings)
            row = {"category": category, "method": method, "ratio": result["ratio"],
                   "KE_percent": 100 * result["ratio"], "c2": c2.tolist(), "c3": c3.tolist(),
                   "saved_ratio": candidate.get("verified_ratio"),
                   "source": "fresh_python_verification", "nfev_linear": result["nfev_linear"],
                   "nfev_nonlinear": result["nfev_nonlinear"], "time_end": result["time_end"],
                   "output_points": result["output_points"], "admissible": True}
            rows.append(row)
            write_json(OUTPUT_ROOT / "fresh_verification.json", rows)
            print(f"{category}, {method}: {row['KE_percent']:.8f}%", flush=True)
    frame = flat_frame(rows)
    frame.to_csv(OUTPUT_ROOT / "fresh_verification.csv", index=False, float_format="%.17g")
    return rows


verification_rows = []
if RECHECK_SELECTED_DESIGNS:
    linear_check = solve_pair(np.zeros(4), np.zeros(4))
    assert abs(linear_check["ratio"] - 1.0) < 1e-10
    verification_rows = validate_candidates(selected_candidates)
    display(pd.DataFrame(verification_rows)[["category", "method", "KE_percent", "saved_ratio", "admissible"]])

comparison = []
for candidate in selected_candidates:
    category = candidate["category"]
    checks = [row["ratio"] for row in verification_rows if row["category"] == category]
    if checks:
        ratio = max(checks)
        source = "fresh RK45/DOP853; conservative maximum"
    elif RUN_MODE == "saved_presentation":
        ratio = candidate["verified_ratio"]
        source = "saved verified candidate; not re-evaluated in this run"
    else:
        ratio = candidate["ratio"]
        source = "new RK45 search; separate verification not run"
    comparison.append({"category": category, "paper_percent": 100 * PAPER_RATIOS[category],
                       "selected_DEM_percent": 100 * ratio, "ratio": ratio,
                       "source": source, "c2": candidate["c2"], "c3": candidate["c3"]})
write_json(OUTPUT_ROOT / "final_designs.json", comparison)
flat_frame(comparison).to_csv(OUTPUT_ROOT / "final_designs.csv", index=False, float_format="%.17g")
display(pd.DataFrame(comparison)[["category", "paper_percent", "selected_DEM_percent", "source"]])

## 13. Independent Local Prediction Test

In [ ]:
def heldout_local_test(category, count=16):
    artifact = final_models[category]
    center, radius, model = artifact["center"], artifact["radius"], artifact["gp"]
    rng = np.random.default_rng(8081)
    X = np.clip(center + rng.normal(size=(count, len(center))) * 0.01, 0, 1)
    train_global = center + radius * model.X_train_
    X = X[np.min(cdist(X, train_global), axis=1) > 1e-8]
    mean, std = predict_gp(model, (X - center) / radius)
    log = DesignLog(OUTPUT_ROOT / "heldout" / category, category,
                   {"stage": "heldout_prediction", "seed": 8081, "count": count, "scale": 0.01})
    actual, prediction = [], []
    for j, z in enumerate(X):
        row = log.evaluate(z, f"test_{j}", "heldout_only", predicted_log_ratio=float(mean[j]))
        actual.append(np.log10(row["ratio"]))
        prediction.append({"actual_ratio": row["ratio"], "predicted_ratio": 10 ** mean[j],
                           "sigma_log10": std[j], "category": category})
    actual = np.array(actual)
    metrics = {"category": category, "n": len(actual), "MAE_log10": mean_absolute_error(actual, mean),
               "R2_log10": r2_score(actual, mean),
               "median_error_factor": float(10 ** np.median(np.abs(actual - mean))),
               "used_for_training": False, "test_distribution": "local Gaussian scale 0.01"}
    write_json(log.directory / "metrics.json", metrics)
    pd.DataFrame(prediction).to_csv(log.directory / "predictions.csv", index=False)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(100 * 10 ** actual, 100 * 10 ** mean)
    limits = np.array([min((10 ** actual).min(), (10 ** mean).min()),
                       max((10 ** actual).max(), (10 ** mean).max())]) * 100
    ax.plot(limits, limits, linestyle="--", label="Equal prediction and simulation")
    ax.set(xlabel="DEM KE ratio (%)", ylabel="GP predicted KE ratio (%)", title=CATEGORY_TITLES[category])
    ax.grid(alpha=0.18)
    ax.legend()
    finish_figure(fig, f"heldout_{category}")
    return metrics

if RUN_PREDICTION_TEST:
    display(pd.DataFrame([heldout_local_test(c["category"]) for c in selected_candidates]))

## 14. First GP Result Visualizations

In [ ]:
def plot_initial_designs(archive):
    first = [r for r in archive if r.get("phase") == "initial_sobol"][:256]
    if not first:
        return
    c2 = np.array([r["c2"][0] for r in first])
    c3 = np.array([r["c3"][0] for r in first])
    target = np.log10([r["ratio"] for r in first])
    fig, ax = plt.subplots(figsize=(7, 5))
    dots = ax.scatter(c2, c3, c=target, s=18, alpha=0.85)
    bar = fig.colorbar(dots, ax=ax)
    bar.set_label("Simulated log10(KE ratio)")
    ax.set(xlabel="Family A quadratic coefficient, c2_A", ylabel="Family A cubic coefficient, c3_A",
           title="Initial joint designs: two-coordinate projection")
    ax.grid(alpha=0.15)
    fig.text(0.5, -0.01, "Each dot contains all eight coefficients. Only family A is displayed.",
             ha="center", fontsize=9)
    finish_figure(fig, "01_initial_sobol_projection")


def region_summary(rows):
    summary = []
    for region in sorted({r["region"] for r in rows}):
        subset = [r for r in rows if r["region"] == region]
        centers = [r for r in subset if r["phase"] == "seed_recheck"]
        if not centers:
            continue
        center = centers[0]
        best = min(subset, key=lambda r: r["ratio"])
        summary.append({"region": region, "initial_percent": 100 * center["ratio"],
                        "best_percent": 100 * best["ratio"], "evaluations": len(subset),
                        "center_z": center["z"]})
    return summary


def plot_region_outcomes(rows):
    summary = region_summary(rows)
    if not summary:
        return
    best_regions = {r["region"] for r in sorted(summary, key=lambda r: r["best_percent"])[:CONTINUE_REGIONS]}
    fig, ax = plt.subplots(figsize=(8, 4.6))
    for i, row in enumerate(summary):
        ax.plot([row["initial_percent"], row["best_percent"]], [i, i], linewidth=1.2)
        ax.scatter(row["initial_percent"], i, marker="o", facecolors="none", s=45,
                   label="Initial center" if i == 0 else None)
        ax.scatter(row["best_percent"], i, marker="s", s=35,
                   label="Best after regional search" if i == 0 else None)
    ax.axvline(100 * PAPER_RATIOS["heterogeneous"], linestyle=":", linewidth=1,
               label="Paper reported reference: 0.56%")
    ax.set_yticks(range(len(summary)), [f"Region {r['region']}" + (" (retained)" if r["region"] in best_regions else "")
                                       for r in summary])
    ax.invert_yaxis()
    ax.set(xlabel="Peak KE ratio (%)", title="Performance across local GP regions")
    ax.grid(axis="x", alpha=0.18)
    ax.legend(fontsize=8)
    finish_figure(fig, "02_regional_outcomes")
    pd.DataFrame([{k: v for k, v in row.items() if k != "center_z"} for row in summary]).to_csv(
        OUTPUT_ROOT / "regional_outcomes.csv", index=False)


def plot_convergence(first_stage, second_stage):
    rows = first_stage + second_stage
    if not rows:
        return
    ratio = np.array([r["ratio"] for r in rows])
    initialization = sum(r["phase"] in ("seed_recheck", "local_initialization") for r in first_stage)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.step(np.arange(1, len(rows) + 1), 100 * np.minimum.accumulate(ratio), where="post",
             label="Best recorded DEM screening result")
    ax.axhline(100 * PAPER_RATIOS["heterogeneous"], linestyle=":", linewidth=1,
                label="Paper reported reference: 0.56%")
    ax.axvline(initialization, linestyle=":", linewidth=1, label=f"Initialization: {initialization} evaluations")
    ax.axvline(len(first_stage), linestyle="--", linewidth=1,
                label=f"Retain {CONTINUE_REGIONS} regions after {len(first_stage)} evaluations")
    ax.set(xlabel="Additional evaluations in the multi-region stage", ylabel="Best-so-far KE ratio (%)",
           title="Multi-region GP optimization convergence")
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8)
    fig.text(0.5, -0.01, "Earlier archive-generation costs are additional. No direct simulator polishing.",
             ha="center", fontsize=9)
    finish_figure(fig, "03_gp_convergence")


def plot_region_pca(rows):
    if len(rows) < 3:
        return
    X = np.array([r["z"] for r in rows])
    projection = PCA(n_components=2)
    coordinates = projection.fit_transform(X)
    summary = region_summary(rows)
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for region in sorted({r["region"] for r in rows}):
        mask = np.array([r["region"] == region for r in rows])
        ax.scatter(coordinates[mask, 0], coordinates[mask, 1], s=9, alpha=0.4, label=f"Region {region}")
    for row in summary:
        position = projection.transform(np.array(row["center_z"])[None])[0]
        ax.scatter(*position, marker="D", s=45, facecolors="none")
        ax.annotate(f"s{row['region']}", position, xytext=(4, 5), textcoords="offset points", fontsize=9)
    explained = projection.explained_variance_ratio_ * 100
    ax.set(xlabel=f"Principal component 1 ({explained[0]:.1f}% of coordinate variance)",
           ylabel=f"Principal component 2 ({explained[1]:.1f}% of coordinate variance)",
           title="Projection of eight-dimensional design coordinates")
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8, markerscale=2)
    fig.text(0.5, -0.01, "PCA is used for display only; all GP fitting and distances use the full input vector.",
             ha="center", fontsize=9)
    finish_figure(fig, "04_region_pca")
    write_json(OUTPUT_ROOT / "pca_summary.json", {"explained_variance_ratio": projection.explained_variance_ratio_,
                                                "mean": projection.mean_, "components": projection.components_})


def plot_final_comparison(rows):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    positions = np.arange(len(rows))
    width = 0.36
    paper = [row["paper_percent"] for row in rows]
    actual = [row["selected_DEM_percent"] for row in rows]
    a = ax.bar(positions - width / 2, paper, width, label="Paper reported reference")
    b = ax.bar(positions + width / 2, actual, width, label="First GP; no direct polishing")
    ax.bar_label(a, fmt="%.4f", padding=3, fontsize=9)
    ax.bar_label(b, fmt="%.4f", padding=3, fontsize=9)
    labels = [CATEGORY_TITLES[row["category"]].replace(" excluding ", "\nexcluding ") for row in rows]
    ax.set_xticks(positions, labels)
    ax.set(ylabel="Peak KE ratio (%)", title="GP-selected spring designs")
    ax.set_ylim(0, max(paper + actual) * 1.25)
    ax.grid(axis="y", alpha=0.18)
    ax.legend(fontsize=9)
    fig.text(0.5, -0.01, "Python DEM results; the notebook does not execute native MATLAB.", ha="center", fontsize=9)
    finish_figure(fig, "05_final_comparison")


if "heterogeneous" in archives:
    plot_initial_designs(archives["heterogeneous"])
if "heterogeneous" in local_records:
    plot_region_outcomes(local_records["heterogeneous"])
    plot_convergence(local_records["heterogeneous"], continuation_rows)
    plot_region_pca(local_records["heterogeneous"])
plot_final_comparison(comparison)

## 15. Export Data, Models, and Figures

In [ ]:
validation_status = {
    "run_mode": RUN_MODE,
    "simulation_source": "included RK45 translation" if EXTERNAL_DEM is None else "uploaded DEM Python.zip",
    "full_search_executed_in_this_run": RUN_MODE != "saved_presentation",
    "saved_archive_used": RUN_MODE == "saved_presentation",
    "sensitivity_regenerated": RUN_SENSITIVITY,
    "selected_candidates_recomputed": RECHECK_SELECTED_DESIGNS,
    "independent_prediction_test_run": RUN_PREDICTION_TEST,
    "optional_next_step_run": RUN_NEXT_STEP,
    "matlab_validation": "Not executed by this notebook; the presentation reports a separate team comparison.",
    "full_search_guarantees_slide_numbers": False,
    "oat_data_used_for_joint_gp_training": False,
}
write_json(OUTPUT_ROOT / "validation_status.json", validation_status)
write_json(OUTPUT_ROOT / "source_manifest.json", SOURCE_MANIFEST)
flat_frame(sensitivity_rows).to_csv(OUTPUT_ROOT / "sensitivity_data.csv", index=False, float_format="%.17g")
for category, rows in local_records.items():
    flat_frame(rows).to_csv(OUTPUT_ROOT / f"six_region_{category}.csv", index=False, float_format="%.17g")
flat_frame(continuation_rows).to_csv(OUTPUT_ROOT / "three_region_continuation.csv", index=False, float_format="%.17g")
archive_path = Path(shutil.make_archive(str(OUTPUT_ROOT.parent / ("Week6_GP_Results_" + RUN_MODE)),
                                      "zip", root_dir=OUTPUT_ROOT))
display(pd.DataFrame([{"output": "Results archive", "path": str(archive_path)},
                      {"output": "Figures", "path": str(FIGURES)}]))
if DOWNLOAD_RESULTS and "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive_path))